# 🔬 Crime Hotspot — Calibration for All Models

Loads pre-trained weights for **ST-Transformer, ConvLSTM, LSTM-only, Conv-only**
and produces the same calibration artefacts as the ST-GAT notebook so the
comparison notebook can pick them all up.

### Artefacts saved per model per city
```
outputs/calibration/{model}/{city}_proba_ts.npy   ← temperature-scaled
outputs/calibration/{model}/{city}_proba_cal.npy  ← Platt-calibrated
outputs/calibration/{model}/{city}_true.npy       ← binary ground truth
outputs/calibration/{model}/{city}_meta.json      ← T_opt, best_threshold
```

### ⚠️ Architecture parameters
The model definitions below use the same hyper-parameters that were used
during training (`hidden=32`, `IN_CH=6`, etc.).  If you trained with
different values, update the **`MODEL_CFG`** dict in Step 2 before loading weights.

## 📁 STEP 1 — Mount Drive & Install Dependencies

In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import subprocess
subprocess.run(['pip', 'install', '-q', 'torch_geometric'], check=True)
subprocess.run(['pip', 'install', '-q', 'sympy==1.13.1'], check=True)

import os, json, warnings, math
import numpy  as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import joblib
warnings.filterwarnings('ignore')

from sklearn.metrics import (
    roc_auc_score, f1_score, precision_score, recall_score,
    accuracy_score, average_precision_score, precision_recall_curve
)
from sklearn.linear_model import LogisticRegression as _LR
from scipy.optimize       import minimize_scalar
from scipy.special        import expit as scipy_expit

BASE_DIR  = '/content/drive/MyDrive/CrimeHotspot'
PROC_DIR  = f'{BASE_DIR}/data/processed'
MODEL_DIR = f'{BASE_DIR}/models'

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {DEVICE}')
print('✅ Setup complete')

Mounted at /content/drive
Device : cpu
✅ Setup complete


## ⚙️ STEP 2 — Config, Model Hyper-Parameters & Shared Helpers

In [2]:
# ── Grid / data config ────────────────────────────────────────────────────────
with open(f'{PROC_DIR}/config.json') as f:
    cfg = json.load(f)

CELL_SIZE = cfg['CELL_SIZE']
SEQ_LEN   = cfg['SEQ_LEN']

# ── Model architecture hyper-parameters ──────────────────────────────────────
# ⚠️  Update these to match your training runs if you used different values.
IN_CH  = 6    # 1 dynamic crime count + 5 static POI/KDE features

MODEL_CFG = {
    'sttransformer': dict(in_ch=IN_CH, hidden=64, n_heads=4, n_layers=2, dropout=0.1),
    'convlstm'     : dict(in_ch=IN_CH, hidden=32, kernel_size=3, dropout=0.1),
    'lstm'         : dict(in_ch=IN_CH, hidden=64, n_layers=2,   dropout=0.1),
    'conv'         : dict(in_ch=IN_CH, hidden=32, kernel_size=3, dropout=0.1),
}

# ── Models to run — set False to skip one ────────────────────────────────────
RUN = {
    'sttransformer': True,
    'convlstm'     : True,
    'lstm'         : True,
    'conv'         : True,
}

# ── Shared metric helpers ─────────────────────────────────────────────────────
def compute_pai(y_true, y_proba, k):
    n_total   = len(y_true)
    n_hotspot = max(float(y_true.sum()), 1)
    k_        = min(k, n_total)
    topk      = np.argsort(y_proba)[-k_:]
    return round((y_true[topk].sum() / n_hotspot) / (k_ / n_total), 4)

def compute_metrics(y_true, y_proba, threshold=0.5):
    y_pred = (y_proba >= threshold).astype(int)
    m = {
        'Accuracy' : round(accuracy_score(y_true, y_pred), 4),
        'AUROC'    : round(roc_auc_score(y_true, y_proba), 4),
        'AUPRC'    : round(average_precision_score(y_true, y_proba), 4),
        'F1'       : round(f1_score(y_true, y_pred, zero_division=0), 4),
        'Precision': round(precision_score(y_true, y_pred, zero_division=0), 4),
        'Recall'   : round(recall_score(y_true, y_pred, zero_division=0), 4),
    }
    for k in [50, 100, 200]:
        m[f'PAI@{k}'] = compute_pai(y_true, y_proba, k)
    return m

def platt_calibrate(y_cal_proba, y_cal_true, y_test_proba):
    platt = _LR(C=1.0, solver='lbfgs', max_iter=1000)
    platt.fit(y_cal_proba.reshape(-1, 1), y_cal_true)
    y_cal_cal  = platt.predict_proba(y_cal_proba.reshape(-1, 1))[:, 1]
    y_test_cal = platt.predict_proba(y_test_proba.reshape(-1, 1))[:, 1]
    prec, rec, th = precision_recall_curve(y_cal_true, y_cal_cal)
    f1s    = 2 * prec * rec / (prec + rec + 1e-9)
    best_t = float(th[np.argmax(f1s)])
    return y_test_cal, best_t, platt

def find_temperature(logits, y_true):
    def nll(T):
        p = scipy_expit(logits / T)
        p = np.clip(p, 1e-7, 1 - 1e-7)
        return -np.mean(y_true * np.log(p) + (1 - y_true) * np.log(1 - p))
    T_opt = minimize_scalar(nll, bounds=(0.1, 20.0), method='bounded').x
    return T_opt, scipy_expit(logits / T_opt)

def build_queen_graph(H, W):
    src, dst = [], []
    for r in range(H):
        for c in range(W):
            node = r * W + c
            for dr in [-1, 0, 1]:
                for dc in [-1, 0, 1]:
                    if dr == 0 and dc == 0: continue
                    nr, nc = r + dr, c + dc
                    if 0 <= nr < H and 0 <= nc < W:
                        src.append(node); dst.append(nr * W + nc)
    return torch.tensor([src, dst], dtype=torch.long)

print('✅ Config & helpers ready')

✅ Config & helpers ready


## 📂 STEP 3 — Load Chicago Data & Build Splits

In [3]:
daily       = pd.read_csv(f'{PROC_DIR}/daily_features.csv', parse_dates=['date_day'])
cell_index  = pd.read_csv(f'{PROC_DIR}/cell_index.csv')
grid_tensor = np.load(f'{PROC_DIR}/grid_tensor.npy')

_, H, W = grid_tensor.shape
N       = H * W
T_total = grid_tensor.shape[0]

split_t   = int(T_total * 0.80)
cal_split = int(split_t  * 0.85)

# ── Static node features ──────────────────────────────────────────────────────
STATIC_COLS = ['poi_bar', 'poi_school', 'poi_transit', 'poi_hospital', 'kde_density']
poi_per_cell = (
    daily.groupby('cell_id')[STATIC_COLS]
         .mean()
         .reindex(cell_index['cell_id'])
         .fillna(0)
)
poi_min  = poi_per_cell.min()
poi_max  = poi_per_cell.max()
poi_norm = (poi_per_cell - poi_min) / (poi_max - poi_min + 1e-9)
node_features = poi_norm.values.astype(np.float32)   # (N, 5)

print(f'Grid     : {grid_tensor.shape}   H={H}  W={W}  N={N}')
print(f'Split    : train=0→{split_t}  cal={cal_split}→{split_t}  test={split_t}→{T_total}')
print('✅ Chicago data loaded')

Grid     : (1547, 85, 86)   H=85  W=86  N=7310
Split    : train=0→1237  cal=1051→1237  test=1237→1547
✅ Chicago data loaded


## 🗂️ STEP 4 — Dataset & DataLoaders

In [4]:
class CrimeGraphDataset(Dataset):
    """Shared dataset — outputs (SEQ_LEN, N, IN_CH) tensors.
    Grid-based models reshape internally using stored H, W."""
    def __init__(self, grid, seq_len, node_feats):
        self.grid       = grid
        self.seq_len    = seq_len
        self.node_feats = node_feats
        T, H_, W_       = grid.shape
        self.N          = H_ * W_

    def __len__(self):
        return len(self.grid) - self.seq_len

    def __getitem__(self, idx):
        x_dyn = self.grid[idx : idx + self.seq_len].astype(np.float32)
        x_dyn = x_dyn.reshape(self.seq_len, self.N, 1)
        max_c = x_dyn.max() + 1e-9
        x_dyn = x_dyn / max_c
        static = np.tile(self.node_feats[np.newaxis], (self.seq_len, 1, 1))
        x = np.concatenate([x_dyn, static], axis=-1)   # (T, N, IN_CH)
        y = (self.grid[idx + self.seq_len] > 0).astype(np.float32).reshape(self.N)
        return torch.tensor(x), torch.tensor(y)


cal_ds  = CrimeGraphDataset(grid_tensor[cal_split:split_t], SEQ_LEN, node_features)
test_ds = CrimeGraphDataset(grid_tensor[split_t:],          SEQ_LEN, node_features)
cal_dl  = DataLoader(cal_ds,  batch_size=4, shuffle=False, num_workers=2)
test_dl = DataLoader(test_ds, batch_size=4, shuffle=False, num_workers=2)

edge_index     = build_queen_graph(H, W)
edge_index_dev = edge_index.to(DEVICE)

xb, yb = next(iter(test_dl))
print(f'Batch x : {xb.shape}  (B, T, N, IN_CH)')
print(f'Batch y : {yb.shape}  (B, N)')
print('✅ DataLoaders ready')

Batch x : torch.Size([4, 7, 7310, 6])  (B, T, N, IN_CH)
Batch y : torch.Size([4, 7310])  (B, N)
✅ DataLoaders ready


## 🧠 STEP 5 — Model Definitions

All four models accept `(B, T, N, F)` input (same as ST-GAT) and return `(B, N)` raw logits.
Grid-based models (ConvLSTM, Conv-only) reshape internally using constructor `H, W` args.

In [5]:
# ─────────────────────────────────────────────────────────────────────────────
# 1. ST-TRANSFORMER
#    Spatial: Multi-head self-attention over N nodes per timestep
#    Temporal: Transformer encoder over T timesteps per node
#    Output:  Linear head → (B, N) logits
# ─────────────────────────────────────────────────────────────────────────────
class STTransformer(nn.Module):
    def __init__(self, in_ch, hidden=64, n_heads=4, n_layers=2, dropout=0.1):
        super().__init__()
        self.input_proj = nn.Linear(in_ch, hidden)

        # Spatial transformer — attends over N nodes at each timestep
        s_layer = nn.TransformerEncoderLayer(
            d_model=hidden, nhead=n_heads, dim_feedforward=hidden * 2,
            dropout=dropout, batch_first=True, norm_first=True)
        self.spatial_transformer = nn.TransformerEncoder(s_layer, num_layers=n_layers)

        # Temporal transformer — attends over T steps per node
        t_layer = nn.TransformerEncoderLayer(
            d_model=hidden, nhead=n_heads, dim_feedforward=hidden * 2,
            dropout=dropout, batch_first=True, norm_first=True)
        self.temporal_transformer = nn.TransformerEncoder(t_layer, num_layers=n_layers)

        self.head = nn.Sequential(
            nn.LayerNorm(hidden),
            nn.Linear(hidden, hidden // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden // 2, 1),
        )

    def forward(self, x_seq, edge_index=None):
        # x_seq : (B, T, N, F)
        B, T, N, nF = x_seq.shape
        x = self.input_proj(x_seq)           # (B, T, N, hidden)

        # Spatial: for each timestep, attend over nodes
        x_sp = x.reshape(B * T, N, -1)
        x_sp = self.spatial_transformer(x_sp)           # (B*T, N, hidden)
        x_sp = x_sp.reshape(B, T, N, -1)

        # Temporal: for each node, attend over time
        x_t = x_sp.permute(0, 2, 1, 3).reshape(B * N, T, -1)  # (B*N, T, hidden)
        x_t = self.temporal_transformer(x_t)                    # (B*N, T, hidden)
        last = x_t[:, -1, :]                                    # (B*N, hidden)

        logits = self.head(last).reshape(B, N)           # (B, N)
        return logits


# ─────────────────────────────────────────────────────────────────────────────
# 2. ConvLSTM
#    Standard ConvLSTM cell — spatial conv in gates, LSTM temporal recurrence
# ─────────────────────────────────────────────────────────────────────────────
class ConvLSTMCell(nn.Module):
    def __init__(self, in_ch, hidden, kernel_size=3):
        super().__init__()
        pad = kernel_size // 2
        self.hidden = hidden
        self.gates = nn.Conv2d(in_ch + hidden, 4 * hidden, kernel_size, padding=pad)

    def forward(self, x, h, c):
        combined = torch.cat([x, h], dim=1)
        gates = self.gates(combined)
        i, f, g, o = gates.chunk(4, dim=1)
        c_ = torch.sigmoid(f) * c + torch.sigmoid(i) * torch.tanh(g)
        h_ = torch.sigmoid(o) * torch.tanh(c_)
        return h_, c_

class ConvLSTM(nn.Module):
    def __init__(self, in_ch, hidden=32, kernel_size=3, dropout=0.1, grid_H=None, grid_W=None):
        super().__init__()
        self.hidden    = hidden
        self.grid_H    = grid_H
        self.grid_W    = grid_W
        # Project IN_CH features to a single channel before ConvLSTM
        self.feat_proj = nn.Conv2d(in_ch, hidden, kernel_size=1)
        self.cell      = ConvLSTMCell(hidden, hidden, kernel_size)
        self.dropout   = nn.Dropout2d(dropout)
        self.head      = nn.Sequential(
            nn.Conv2d(hidden, hidden // 2, kernel_size=1),
            nn.ReLU(),
            nn.Dropout2d(dropout),
            nn.Conv2d(hidden // 2, 1, kernel_size=1),
        )

    def forward(self, x_seq, edge_index=None):
        # x_seq : (B, T, N, F)  —  reshape to (B, T, F, H, W)
        B, T, N, F = x_seq.shape
        gH = self.grid_H; gW = self.grid_W
        x = x_seq.reshape(B, T, gH, gW, F).permute(0, 1, 4, 2, 3)  # (B,T,F,H,W)

        h = torch.zeros(B, self.hidden, gH, gW, device=x_seq.device)
        c = torch.zeros(B, self.hidden, gH, gW, device=x_seq.device)
        for t in range(T):
            xt = self.feat_proj(x[:, t])    # (B, hidden, H, W)
            h, c = self.cell(xt, h, c)
            h = self.dropout(h)

        logits = self.head(h)               # (B, 1, H, W)
        return logits.reshape(B, N)         # (B, N)


# ─────────────────────────────────────────────────────────────────────────────
# 3. LSTM-only  (no spatial processing — each node is an independent sequence)
# ─────────────────────────────────────────────────────────────────────────────
class LSTMOnly(nn.Module):
    def __init__(self, in_ch, hidden=64, n_layers=2, dropout=0.1):
        super().__init__()
        self.lstm = nn.LSTM(in_ch, hidden, num_layers=n_layers,
                            batch_first=True, dropout=dropout if n_layers > 1 else 0)
        self.head = nn.Sequential(
            nn.LayerNorm(hidden),
            nn.Linear(hidden, hidden // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden // 2, 1),
        )

    def forward(self, x_seq, edge_index=None):
        # x_seq : (B, T, N, F)  →  process each node as a separate sequence
        B, T, N, F = x_seq.shape
        x = x_seq.permute(0, 2, 1, 3).reshape(B * N, T, F)  # (B*N, T, F)
        out, _ = self.lstm(x)          # (B*N, T, hidden)
        last   = out[:, -1, :]         # (B*N, hidden)
        logits = self.head(last).reshape(B, N)
        return logits


# ─────────────────────────────────────────────────────────────────────────────
# 4. Conv-only  (no temporal recurrence — mean-pool across time then Conv2D)
# ─────────────────────────────────────────────────────────────────────────────
class ConvOnly(nn.Module):
    def __init__(self, in_ch, hidden=32, kernel_size=3, dropout=0.1, grid_H=None, grid_W=None):
        super().__init__()
        self.grid_H = grid_H
        self.grid_W = grid_W
        pad = kernel_size // 2
        # Temporal aggregation: collapse T with a learned 1D conv over time
        self.temporal_pool = nn.Conv1d(in_ch, in_ch, kernel_size=1, groups=in_ch)
        self.spatial = nn.Sequential(
            nn.Conv2d(in_ch,      hidden,     kernel_size, padding=pad), nn.BatchNorm2d(hidden),  nn.ReLU(),
            nn.Conv2d(hidden,     hidden,     kernel_size, padding=pad), nn.BatchNorm2d(hidden),  nn.ReLU(),
            nn.Conv2d(hidden,     hidden * 2, kernel_size, padding=pad), nn.BatchNorm2d(hidden*2),nn.ReLU(),
            nn.Dropout2d(dropout),
            nn.Conv2d(hidden * 2, 1, kernel_size=1),
        )

    def forward(self, x_seq, edge_index=None):
        # x_seq : (B, T, N, F)
        B, T, N, F = x_seq.shape
        gH = self.grid_H; gW = self.grid_W
        # Mean-pool across time → (B, N, F)
        x = x_seq.mean(dim=1)                               # (B, N, F)
        x = x.permute(0, 2, 1)                              # (B, F, N)
        x = self.temporal_pool(x).permute(0, 2, 1)          # (B, N, F)
        x = x.reshape(B, gH, gW, F).permute(0, 3, 1, 2)    # (B, F, H, W)
        logits = self.spatial(x)                            # (B, 1, H, W)
        return logits.reshape(B, N)


print('✅ All model classes defined')

✅ All model classes defined


## 🔩 STEP 6 — Instantiate & Load Pre-Trained Weights

In [6]:
# ── Inspect convlstm.pt to reverse-engineer the original architecture ─────────
ckpt = torch.load(f'{MODEL_DIR}/convlstm.pt', map_location='cpu')
print('ConvLSTM checkpoint keys & shapes:')
for k, v in ckpt.items():
    print(f'  {k:<35} {tuple(v.shape)}')

ConvLSTM checkpoint keys & shapes:
  cell.conv.weight                    (64, 17, 3, 3)
  cell.conv.bias                      (64,)
  head.0.weight                       (1, 16, 1, 1)
  head.0.bias                         (1,)


In [7]:
# ── CELL B — Corrected ConvLSTM (exactly matching checkpoint) ─────────────────

class ConvLSTMCell(nn.Module):
    def __init__(self, in_ch, hidden, kernel_size=3):
        super().__init__()
        pad = kernel_size // 2
        self.hidden = hidden
        self.conv = nn.Conv2d(in_ch + hidden, 4 * hidden, kernel_size, padding=pad)

    def forward(self, x, h, c):
        gates = self.conv(torch.cat([x, h], dim=1))
        i, f, g, o = gates.chunk(4, dim=1)
        c_ = torch.sigmoid(f) * c + torch.sigmoid(i) * torch.tanh(g)
        h_ = torch.sigmoid(o) * torch.tanh(c_)
        return h_, c_


class ConvLSTM(nn.Module):
    def __init__(self, in_ch=1, hidden=16, kernel_size=3, dropout=0.1,
                 grid_H=None, grid_W=None):
        super().__init__()
        self.hidden = hidden
        self.in_ch  = in_ch     # store so forward can slice correctly
        self.grid_H = grid_H
        self.grid_W = grid_W
        self.cell   = ConvLSTMCell(in_ch, hidden, kernel_size)
        self.dropout = nn.Dropout2d(dropout)
        self.head    = nn.Sequential(
            nn.Conv2d(hidden, 1, kernel_size=1)   # matches head.0 shape (1,16,1,1)
        )

    def forward(self, x_seq, edge_index=None):
        B, T, N, F = x_seq.shape
        gH, gW = self.grid_H, self.grid_W
        # Slice only the first in_ch features (dynamic crime count)
        # — matches training where only 1 feature was used
        x = x_seq[..., :self.in_ch]                          # (B, T, N, in_ch)
        x = x.reshape(B, T, gH, gW, self.in_ch)
        x = x.permute(0, 1, 4, 2, 3)                        # (B, T, in_ch, H, W)

        h = torch.zeros(B, self.hidden, gH, gW, device=x_seq.device)
        c = torch.zeros(B, self.hidden, gH, gW, device=x_seq.device)
        for t in range(T):
            h, c = self.cell(x[:, t], h, c)
            h = self.dropout(h)

        return self.head(h).reshape(B, N)                    # (B, N)


class STTransformer(nn.Module):
    def __init__(self, in_ch=1, hidden=64, n_heads=3, n_layers=2,
                 dropout=0.1, grid_H=None, grid_W=None, seq_len=7, n_patches=121):
        super().__init__()
        self.grid_H    = grid_H
        self.grid_W    = grid_W
        self.n_patches = n_patches        # 121 = 11×11
        self.patch_hw  = int(n_patches**0.5)   # 11
        self.seq_len   = seq_len
        self.in_ch     = in_ch            # 1 — only dynamic crime count

        # ── CNN Encoder ───────────────────────────────────────────────────────
        # encoder.0, encoder.2 in checkpoint (ReLU at .1 and .3 have no params)
        self.encoder = nn.Sequential(
            nn.Conv2d(1,  32, kernel_size=3, padding=1),   # encoder.0
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),   # encoder.2
            nn.ReLU(),
        )
        # AdaptiveAvgPool has no params — not in state_dict, keep separate
        self.pool = nn.AdaptiveAvgPool2d((self.patch_hw, self.patch_hw))

        # ── Positional Embeddings ─────────────────────────────────────────────
        self.spatial_pos  = nn.Embedding(n_patches, hidden)   # (121, 64)
        self.temporal_pos = nn.Embedding(seq_len,   hidden)   # (7,   64)

        # ── Spatial Transformer (n_heads=3 → in_proj=192) ─────────────────────
        s_layer = nn.TransformerEncoderLayer(
            d_model=hidden, nhead=n_heads, dim_feedforward=hidden * 4,
            dropout=dropout, batch_first=True)
        self.spatial_transformer = nn.TransformerEncoder(s_layer, num_layers=n_layers)

        # ── Temporal Transformer ──────────────────────────────────────────────
        t_layer = nn.TransformerEncoderLayer(
            d_model=hidden, nhead=n_heads, dim_feedforward=hidden * 4,
            dropout=dropout, batch_first=True)
        self.temporal_transformer = nn.TransformerEncoder(t_layer, num_layers=n_layers)

        # ── CNN Decoder ───────────────────────────────────────────────────────
        # decoder.0, decoder.2, decoder.4 (ReLU at .1 and .3 have no params)
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1),  # decoder.0
            nn.ReLU(),
            nn.ConvTranspose2d(32, 16, kernel_size=4, stride=2, padding=1),  # decoder.2
            nn.ReLU(),
            nn.ConvTranspose2d(16,  1, kernel_size=4, stride=2, padding=1),  # decoder.4
        )

    def forward(self, x_seq, edge_index=None):
        B, T, N, nF = x_seq.shape
        gH, gW = self.grid_H, self.grid_W
        ph     = self.patch_hw

        # Use only dynamic crime count feature (in_ch=1)
        x = x_seq[..., :self.in_ch]                                    # (B, T, N, 1)
        x = x.reshape(B * T, 1, gH, gW)                               # (B*T, 1, H, W)

        # ── Encode ────────────────────────────────────────────────────────────
        x = self.encoder(x)                                            # (B*T, 64, H, W)
        x = self.pool(x)                                               # (B*T, 64, 11, 11)

        # ── Spatial Transformer ───────────────────────────────────────────────
        sp_idx = torch.arange(self.n_patches, device=x_seq.device)
        x = x.flatten(2).permute(0, 2, 1)                             # (B*T, 121, 64)
        x = x + self.spatial_pos(sp_idx)
        x = self.spatial_transformer(x)                                # (B*T, 121, 64)

        # ── Temporal Transformer ──────────────────────────────────────────────
        t_idx = torch.arange(T, device=x_seq.device)
        x = x.reshape(B, T, self.n_patches, -1)
        x = x.permute(0, 2, 1, 3).reshape(B * self.n_patches, T, -1) # (B*121, T, 64)
        x = x + self.temporal_pos(t_idx)
        x = self.temporal_transformer(x)                               # (B*121, T, 64)

        # Take final timestep, reshape back to spatial grid
        x = x[:, -1, :]                                                # (B*121, 64)
        x = x.reshape(B, self.n_patches, -1)
        x = x.permute(0, 2, 1).reshape(B, 64, ph, ph)                 # (B, 64, 11, 11)

        # ── Decode ────────────────────────────────────────────────────────────
        x = self.decoder(x)                                            # (B, 1, ~88, ~88)
        # Interpolate to exact Chicago grid size
        x = F.interpolate(x, size=(gH, gW), mode='bilinear', align_corners=False)
        return x.reshape(B, N)                                         # (B, N)


class LSTMOnly(nn.Module):
    def __init__(self, in_ch, hidden=64, n_layers=2, dropout=0.1):
        super().__init__()
        self.lstm = nn.LSTM(in_ch, hidden, num_layers=n_layers,
                            batch_first=True,
                            dropout=dropout if n_layers > 1 else 0)
        self.head = nn.Sequential(
            nn.LayerNorm(hidden),
            nn.Linear(hidden, hidden // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden // 2, 1),
        )

    def forward(self, x_seq, edge_index=None):
        B, T, N, F = x_seq.shape
        x = x_seq.permute(0, 2, 1, 3).reshape(B * N, T, F)
        out, _ = self.lstm(x)
        return self.head(out[:, -1, :]).reshape(B, N)


class ConvOnly(nn.Module):
    def __init__(self, in_ch, hidden=32, kernel_size=3, dropout=0.1,
                 grid_H=None, grid_W=None):
        super().__init__()
        self.grid_H = grid_H
        self.grid_W = grid_W
        pad = kernel_size // 2
        self.spatial = nn.Sequential(
            nn.Conv2d(in_ch,      hidden,     kernel_size, padding=pad),
            nn.BatchNorm2d(hidden),   nn.ReLU(),
            nn.Conv2d(hidden,     hidden,     kernel_size, padding=pad),
            nn.BatchNorm2d(hidden),   nn.ReLU(),
            nn.Conv2d(hidden,     hidden * 2, kernel_size, padding=pad),
            nn.BatchNorm2d(hidden*2), nn.ReLU(),
            nn.Dropout2d(dropout),
            nn.Conv2d(hidden * 2, 1, kernel_size=1),
        )

    def forward(self, x_seq, edge_index=None):
        B, T, N, F = x_seq.shape
        gH, gW = self.grid_H, self.grid_W
        x = x_seq[..., :_c1_in].mean(dim=1).reshape(B, gH, gW, _c1_in).permute(0, 3, 1, 2)
        return self.spatial(x).reshape(B, N)


print('✅ All model architectures defined')

✅ All model architectures defined


In [8]:
for fname in ['st_transformer_final.pt', 'st_transformer_best.pt']:
    ckpt = torch.load(f'{MODEL_DIR}/{fname}', map_location='cpu')
    print(f'\n── {fname} ──')
    for k, v in ckpt.items():
        print(f'  {k:<45} {tuple(v.shape)}')


── st_transformer_final.pt ──
  encoder.0.weight                              (32, 1, 3, 3)
  encoder.0.bias                                (32,)
  encoder.2.weight                              (64, 32, 3, 3)
  encoder.2.bias                                (64,)
  spatial_pos.weight                            (121, 64)
  temporal_pos.weight                           (7, 64)
  spatial_transformer.layers.0.self_attn.in_proj_weight (192, 64)
  spatial_transformer.layers.0.self_attn.in_proj_bias (192,)
  spatial_transformer.layers.0.self_attn.out_proj.weight (64, 64)
  spatial_transformer.layers.0.self_attn.out_proj.bias (64,)
  spatial_transformer.layers.0.linear1.weight   (256, 64)
  spatial_transformer.layers.0.linear1.bias     (256,)
  spatial_transformer.layers.0.linear2.weight   (64, 256)
  spatial_transformer.layers.0.linear2.bias     (64,)
  spatial_transformer.layers.0.norm1.weight     (64,)
  spatial_transformer.layers.0.norm1.bias       (64,)
  spatial_transformer.layers.0.norm

In [9]:
# ── STEP 6 — Inspect Checkpoints & Load All Pre-Trained Weights ─────────────

import gc

def free_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

# ── 6a: Inspect lstmonly.pt & convonly.pt to auto-detect exact architecture ──
lstm_ckpt = torch.load(f'{MODEL_DIR}/lstmonly.pt', map_location='cpu')
conv_ckpt = torch.load(f'{MODEL_DIR}/convonly.pt', map_location='cpu')

print('lstmonly.pt:')
for k, v in lstm_ckpt.items(): print(f'  {k:<40} {tuple(v.shape)}')
print('\nconvonly.pt:')
for k, v in conv_ckpt.items(): print(f'  {k:<40} {tuple(v.shape)}')

# ── 6b: Corrected LSTMOnly (auto-derived from checkpoint) ────────────────────
# lstm.weight_ih_l0: [4*hidden, N]  →  model was trained with full grid as input
# head.weight, head.bias            →  plain nn.Linear, NOT a Sequential
_lstm_in   = lstm_ckpt['lstm.weight_ih_l0'].shape[1]
_lstm_hid  = lstm_ckpt['lstm.weight_ih_l0'].shape[0] // 4
_lstm_lays = sum(1 for k in lstm_ckpt if k.startswith('lstm.weight_ih'))
print(f'\nLSTMOnly: in={_lstm_in}, hidden={_lstm_hid}, n_layers={_lstm_lays}')

class LSTMOnly(nn.Module):
    """Global-temporal LSTM — all N grid cells as input features per timestep."""
    def __init__(self, n_nodes, hidden=32, n_layers=1):
        super().__init__()
        self.lstm = nn.LSTM(n_nodes, hidden, num_layers=n_layers, batch_first=True)
        self.head = nn.Linear(hidden, n_nodes)

    def forward(self, x_seq, edge_index=None):
        B, T, N_nodes, F = x_seq.shape
        x = x_seq[..., 0]               # (B, T, N) — crime count feature only
        out, _ = self.lstm(x)
        return self.head(out[:, -1, :]) # (B, N)

# ── 6c: Corrected ConvOnly (auto-derived from checkpoint) ────────────────────
# conv1, conv2 are named attributes (not inside self.spatial Sequential)
# head.0 is a single Conv2d or Linear
_c1_in   = conv_ckpt['conv1.weight'].shape[1]
_c1_out  = conv_ckpt['conv1.weight'].shape[0]
_c2_out  = conv_ckpt['conv2.weight'].shape[0]
_head_w  = conv_ckpt['head.0.weight']
_is_conv = (_head_w.dim() == 4)
print(f'ConvOnly: in_ch={_c1_in}, hidden1={_c1_out}, hidden2={_c2_out}, '
      f'head={"Conv2d" if _is_conv else "Linear"}')

class ConvOnly(nn.Module):
    def __init__(self, in_ch, hidden1, hidden2, grid_H=None, grid_W=None):
        super().__init__()
        self.grid_H = grid_H
        self.grid_W = grid_W
        self.conv1 = nn.Conv2d(in_ch,   hidden1, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(hidden1, hidden2, kernel_size=3, padding=1)
        if _is_conv:
            _hk = _head_w.shape[2]
            self.head = nn.Sequential(nn.Conv2d(hidden2, 1, kernel_size=_hk,
                                                 padding=_hk // 2))
        else:
            self.head = nn.Sequential(nn.Linear(_head_w.shape[1], _head_w.shape[0]))

    def forward(self, x_seq, edge_index=None):
        B, T, N, F = x_seq.shape
        gH, gW = self.grid_H, self.grid_W
        x = x_seq[..., :_c1_in].mean(dim=1).reshape(B, gH, gW, _c1_in).permute(0, 3, 1, 2)
        x = torch.relu(self.conv1(x))
        x = torch.relu(self.conv2(x))
        return self.head(x).reshape(B, N)

# ── 6d: Load all four models ──────────────────────────────────────────────────
PT_FILES = {
    'convlstm'     : 'convlstm.pt',
    'sttransformer': 'sttransformer.pt',
    'lstm'         : 'lstmonly.pt',
    'conv'         : 'convonly.pt',
}

SPECS = {
    'convlstm'     : (ConvLSTM,      dict(in_ch=1,          hidden=16,        kernel_size=3,
                                          dropout=0.1,        grid_H=H,         grid_W=W)),
    'sttransformer': (STTransformer, dict(in_ch=1,          hidden=64,        n_heads=4,
                                          n_layers=2,         dropout=0.1,
                                          grid_H=H,           grid_W=W,
                                          seq_len=SEQ_LEN,    n_patches=121)),
    'lstm'         : (LSTMOnly,      dict(n_nodes=_lstm_in, hidden=_lstm_hid, n_layers=_lstm_lays)),
    'conv'         : (ConvOnly,      dict(in_ch=_c1_in,     hidden1=_c1_out,  hidden2=_c2_out,
                                          grid_H=H,           grid_W=W)),
}

MODELS = {}
for name, (cls, kwargs) in SPECS.items():
    pt_path = f'{MODEL_DIR}/{PT_FILES[name]}'
    if not os.path.exists(pt_path):
        print(f'  ⚠️  {name}: weights not found at {pt_path} — skipping')
        continue
    try:
        free_gpu()
        m = cls(**kwargs).to(DEVICE)
        m.load_state_dict(torch.load(pt_path, map_location=DEVICE))
        m.eval()
        n_p = sum(p.numel() for p in m.parameters() if p.requires_grad)
        MODELS[name] = m
        print(f'  ✅ {name:<15} loaded from {PT_FILES[name]}  ({n_p:,} params)')
    except RuntimeError as e:
        print(f'  ❌ {name}: weight mismatch\n     {e}')

print(f'\n{len(MODELS)}/{len(SPECS)} models loaded → proceed to Step 7')

lstmonly.pt:
  lstm.weight_ih_l0                        (128, 7310)
  lstm.weight_hh_l0                        (128, 32)
  lstm.bias_ih_l0                          (128,)
  lstm.bias_hh_l0                          (128,)
  head.weight                              (7310, 32)
  head.bias                                (7310,)

convonly.pt:
  conv1.weight                             (16, 1, 3, 3)
  conv1.bias                               (16,)
  conv2.weight                             (16, 16, 3, 3)
  conv2.bias                               (16,)
  head.0.weight                            (1, 16, 1, 1)
  head.0.bias                              (1,)

LSTMOnly: in=7310, hidden=32, n_layers=1
ConvOnly: in_ch=1, hidden1=16, hidden2=16, head=Conv2d
  ✅ convlstm        loaded from convlstm.pt  (9,873 params)
  ✅ sttransformer   loaded from sttransformer.pt  (268,209 params)
  ✅ lstm            loaded from lstmonly.pt  (1,181,262 params)
  ✅ conv            loaded from convonly.pt  (2,497 pa

In [ ]:
def load_model(name):
    """Instantiate model and load saved weights from MODEL_DIR/{name}.pt"""
    cfg = MODEL_CFG[name]
    if name == 'sttransformer':
        model = STTransformer(**cfg)
    elif name == 'convlstm':
        model = ConvLSTM(**cfg, grid_H=H, grid_W=W)
    elif name == 'lstm':
        model = LSTMOnly(**cfg)
    elif name == 'conv':
        model = ConvOnly(**cfg, grid_H=H, grid_W=W)
    else:
        raise ValueError(f'Unknown model: {name}')

    pt_path = f'{MODEL_DIR}/{name}.pt'
    if not os.path.exists(pt_path):
        raise FileNotFoundError(f'Weights not found: {pt_path}')

    state = torch.load(pt_path, map_location=DEVICE)
    model.load_state_dict(state)
    model = model.to(DEVICE)
    model.eval()
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'  ✅ {name:<15} loaded  ({n_params:,} params)')
    return model


MODELS = {}
for name in ['sttransformer', 'convlstm', 'lstm', 'conv']:
    if not RUN[name]:
        print(f'  ⏭️  {name} — skipped (RUN=False)')
        continue
    try:
        MODELS[name] = load_model(name)
    except FileNotFoundError as e:
        print(f'  ⚠️  {e}')
    except RuntimeError as e:
        print(f'  ❌ {name} weight mismatch — check MODEL_CFG: {e}')

print(f'\n{len(MODELS)} model(s) loaded successfully.')

  ⚠️  Weights not found: /content/drive/MyDrive/CrimeHotspot/models/sttransformer.pt
  ❌ convlstm weight mismatch — check MODEL_CFG: Error(s) in loading state_dict for ConvLSTM:
	Missing key(s) in state_dict: "feat_proj.weight", "feat_proj.bias", "cell.gates.weight", "cell.gates.bias", "head.3.weight", "head.3.bias". 
	Unexpected key(s) in state_dict: "cell.conv.weight", "cell.conv.bias". 
	size mismatch for head.0.weight: copying a param with shape torch.Size([1, 16, 1, 1]) from checkpoint, the shape in current model is torch.Size([16, 32, 1, 1]).
	size mismatch for head.0.bias: copying a param with shape torch.Size([1]) from checkpoint, the shape in current model is torch.Size([16]).
  ⚠️  Weights not found: /content/drive/MyDrive/CrimeHotspot/models/lstm.pt
  ⚠️  Weights not found: /content/drive/MyDrive/CrimeHotspot/models/conv.pt

0 model(s) loaded successfully.


## 🔬 STEP 7 — Chicago Calibration for All Models

In [10]:
@torch.no_grad()
def run_inference(model, loader, apply_sigmoid=False):
    """Generic inference — works for all model types."""
    model.eval()
    preds, trues = [], []
    for xb, yb in loader:
        xb = xb.to(DEVICE).float()
        logits = model(xb, edge_index_dev)  # edge_index ignored by non-graph models
        out = torch.sigmoid(logits) if apply_sigmoid else logits
        preds.append(out.cpu().numpy())
        trues.append(yb.numpy())
    return np.concatenate(preds).ravel(), np.concatenate(trues).ravel()


def calibrate_and_save(model, name, cal_dl, test_dl, city,
                       la_edge_index_dev=None):
    """Full temperature + Platt calibration pipeline, mirrors STGAT notebook."""
    cal_dir = f'{BASE_DIR}/outputs/calibration/{name}'
    os.makedirs(cal_dir, exist_ok=True)

    # ── Step 1: collect calibration set logits ────────────────────────────────
    y_cal_logits, y_cal_t = run_inference(model, cal_dl, apply_sigmoid=False)

    # ── Step 2: temperature scaling on calibration set ────────────────────────
    T_opt, y_cal_ts = find_temperature(y_cal_logits, y_cal_t)

    # ── Step 3: collect test set logits & apply temperature ───────────────────
    y_test_logits, y_test_t = run_inference(model, test_dl, apply_sigmoid=False)
    y_test_raw = scipy_expit(y_test_logits)           # raw sigmoid
    y_test_ts  = scipy_expit(y_test_logits / T_opt)  # temp-scaled

    # ── Step 4: Platt scaling ─────────────────────────────────────────────────
    y_test_cal, best_t, platt = platt_calibrate(y_cal_ts, y_cal_t, y_test_ts)

    # ── Metrics ───────────────────────────────────────────────────────────────
    m_raw = compute_metrics(y_test_t, y_test_ts,  threshold=0.5)
    m_cal = compute_metrics(y_test_t, y_test_cal, threshold=best_t)

    print(f'  T_opt={T_opt:.4f}   best_threshold={best_t:.4f}')
    print(f'  AUROC  raw={m_raw["AUROC"]}  cal={m_cal["AUROC"]}')
    print(f'  AUPRC  raw={m_raw["AUPRC"]}  cal={m_cal["AUPRC"]}')
    print(f'  F1     raw={m_raw["F1"]}     cal={m_cal["F1"]}')

    # ── Save artefacts ────────────────────────────────────────────────────────
    np.save(f'{cal_dir}/{city}_proba_raw.npy', y_test_raw)
    np.save(f'{cal_dir}/{city}_proba_ts.npy',  y_test_ts)
    np.save(f'{cal_dir}/{city}_proba_cal.npy', y_test_cal)
    np.save(f'{cal_dir}/{city}_true.npy',      y_test_t)
    joblib.dump(platt, f'{cal_dir}/{city}_platt_scaler.pkl')
    with open(f'{cal_dir}/{city}_meta.json', 'w') as f:
        json.dump({'T_opt': float(T_opt), 'best_threshold': float(best_t)}, f, indent=2)

    return m_raw, m_cal


# ── Run Chicago calibration for all loaded models ─────────────────────────────
print('=' * 60)
print('  CHICAGO CALIBRATION')
print('=' * 60)

chi_results = {}
for name, model in MODELS.items():
    print(f'\n── {name} ──')
    try:
        m_raw, m_cal = calibrate_and_save(model, name, cal_dl, test_dl, 'chicago')
        chi_results[name] = {'raw': m_raw, 'cal': m_cal}
        print(f'  ✅ Chicago artefacts saved → outputs/calibration/{name}/')
    except Exception as e:
        print(f'  ❌ Failed: {e}')

print('\n✅ Chicago calibration complete')

  CHICAGO CALIBRATION

── convlstm ──
  T_opt=0.1000   best_threshold=0.0739
  AUROC  raw=0.8637  cal=0.8637
  AUPRC  raw=0.2968  cal=0.2968
  F1     raw=0.1231     cal=0.363
  ✅ Chicago artefacts saved → outputs/calibration/convlstm/

── sttransformer ──
  T_opt=20.0000   best_threshold=0.0550
  AUROC  raw=0.5607  cal=0.4393
  AUPRC  raw=0.0944  cal=0.0557
  F1     raw=0.1141     cal=0.1142
  ✅ Chicago artefacts saved → outputs/calibration/sttransformer/

── lstm ──
  T_opt=0.9291   best_threshold=0.1140
  AUROC  raw=0.9341  cal=0.9341
  AUPRC  raw=0.4308  cal=0.4308
  F1     raw=0.2783     cal=0.4622
  ✅ Chicago artefacts saved → outputs/calibration/lstm/

── conv ──
  T_opt=2.1790   best_threshold=0.1869
  AUROC  raw=0.9148  cal=0.9148
  AUPRC  raw=0.3546  cal=0.3546
  F1     raw=0.3655     cal=0.3981
  ✅ Chicago artefacts saved → outputs/calibration/conv/

✅ Chicago calibration complete


## 📊 STEP 8 — Chicago Summary Table

In [11]:
SHOW_COLS = ['AUROC', 'AUPRC', 'F1', 'Precision', 'Recall', 'PAI@100', 'PAI@200']

for cond, label in [('raw', 'Raw (temp-scaled, thr=0.5)'), ('cal', 'Calibrated (Platt + F1-thr)')]:
    rows = []
    for name, res in chi_results.items():
        row = {'Model': name}
        row.update({k: res[cond][k] for k in SHOW_COLS})
        rows.append(row)
    df = pd.DataFrame(rows).set_index('Model')
    print(f'\n===== Chicago — {label} =====')
    print(df.to_string())


===== Chicago — Raw (temp-scaled, thr=0.5) =====
                AUROC   AUPRC      F1  Precision  Recall    PAI@100    PAI@200
Model                                                                         
convlstm       0.8637  0.2968  0.1231     0.4773  0.0707  13.536000  13.536000
sttransformer  0.5607  0.0944  0.1141     0.0615  0.7851   5.777500   5.364900
lstm           0.9341  0.4308  0.2783     0.5447  0.1869  16.177099  16.177099
conv           0.9148  0.3546  0.3655     0.2305  0.8831  15.681900  15.186700

===== Chicago — Calibrated (Platt + F1-thr) =====
                AUROC   AUPRC      F1  Precision  Recall    PAI@100    PAI@200
Model                                                                         
convlstm       0.8637  0.2968  0.3630     0.2937  0.4750  13.536000  13.536000
sttransformer  0.4393  0.0557  0.1142     0.0606  1.0000   0.165100   0.082500
lstm           0.9341  0.4308  0.4622     0.3760  0.5997  16.177099  16.177099
conv           0.9148  0.3546 

## 🌆 STEP 9 — LA Data Preprocessing

In [12]:
LA_FILE = f'{BASE_DIR}/data/raw/la_crimes.csv'

if not os.path.exists(LA_FILE):
    print('⚠️  la_crimes.csv not found — skipping LA calibration')
    HAS_LA = False
else:
    HAS_LA = True
    la_raw = pd.read_csv(LA_FILE, low_memory=False)
    la = la_raw.copy()
    la.columns = [c.strip().lower().replace(' ', '_') for c in la.columns]

    date_col        = 'date_occ' if 'date_occ' in la.columns else 'date_rptd'
    la['date']      = pd.to_datetime(la[date_col], infer_datetime_format=True, errors='coerce')
    la['latitude']  = pd.to_numeric(la.get('lat',  la.get('location_1')), errors='coerce')
    la['longitude'] = pd.to_numeric(la.get('lon',  la.get('location_1')), errors='coerce')
    la.dropna(subset=['date', 'latitude', 'longitude'], inplace=True)
    la = la[(la['latitude'].between(33.7, 34.35)) &
            (la['longitude'].between(-118.65, -118.15))]

    la['grid_row']  = ((la['latitude']  - 33.70)     / CELL_SIZE).astype(int)
    la['grid_col']  = ((la['longitude'] - (-118.65)) / CELL_SIZE).astype(int)
    la['date_only'] = la['date'].dt.date

    la_H = int(la['grid_row'].clip(lower=0).max()) + 1
    la_W = int(la['grid_col'].clip(lower=0).max()) + 1
    la_N = la_H * la_W

    la_counts = la.groupby(['date_only','grid_row','grid_col']).size().reset_index(name='cnt')
    la_dates  = pd.date_range(la['date'].min().date(), la['date'].max().date(), freq='D')
    la_d_idx  = {d.date(): i for i, d in enumerate(la_dates)}
    la_grid   = np.zeros((len(la_dates), la_H, la_W), dtype=np.float32)
    for _, row in la_counts.iterrows():
        d_i = la_d_idx.get(row['date_only'])
        r, c = int(row['grid_row']), int(row['grid_col'])
        if d_i is not None and 0 <= r < la_H and 0 <= c < la_W:
            la_grid[d_i, r, c] = row['cnt']

    la_T         = la_grid.shape[0]
    la_cal_split = int(la_T * 0.70)
    la_split     = int(la_T * 0.80)

    # Node features — zero if no LA POI data
    LA_POI_PATH = f'{PROC_DIR}/la_poi_per_cell.csv'
    if os.path.exists(LA_POI_PATH):
        la_poi = pd.read_csv(LA_POI_PATH)
        la_poi_grid = np.zeros((la_H, la_W, len(STATIC_COLS)), dtype=np.float32)
        for _, row in la_poi.iterrows():
            r, c = int(row.get('grid_row', 0)), int(row.get('grid_col', 0))
            if 0 <= r < la_H and 0 <= c < la_W:
                for fi, col in enumerate(STATIC_COLS):
                    if col in row: la_poi_grid[r, c, fi] = row[col]
        la_node_features = la_poi_grid.reshape(la_N, len(STATIC_COLS)).astype(np.float32)
        la_node_features = np.clip(
            (la_node_features - poi_min.values) / (poi_max.values - poi_min.values + 1e-9), 0, 1)
        print('✅ LA POI loaded')
    else:
        la_node_features = np.zeros((la_N, len(STATIC_COLS)), dtype=np.float32)
        print('⚠️  LA POI not found — using zero static features')

    la_edge_index_dev = build_queen_graph(la_H, la_W).to(DEVICE)

    la_cal_ds  = CrimeGraphDataset(la_grid[la_cal_split:la_split], SEQ_LEN, la_node_features)
    la_test_ds = CrimeGraphDataset(la_grid[la_split:],             SEQ_LEN, la_node_features)
    la_cal_dl  = DataLoader(la_cal_ds,  batch_size=4, shuffle=False, num_workers=2)
    la_test_dl = DataLoader(la_test_ds, batch_size=4, shuffle=False, num_workers=2)

    print(f'LA grid : {la_grid.shape}   H={la_H}  W={la_W}')
    print(f'LA cal  : {len(la_cal_ds)} samples  |  LA test: {len(la_test_ds)} samples')
    print('✅ LA preprocessing done')

✅ LA POI loaded
LA grid : (367, 126, 99)   H=126  W=99
LA cal  : 30 samples  |  LA test: 67 samples
✅ LA preprocessing done


## 🌆 STEP 10 — LA Calibration for All Models

In [14]:
if not HAS_LA:
    print('⚠️  Skipping — no LA data')
else:
    def get_la_model(name, base_model):
        """Rebuild grid-based models with LA dims; skip models tied to Chicago N."""
        if name == 'convlstm':
            la_m = ConvLSTM(in_ch=1, hidden=16, kernel_size=3,
                            dropout=0.1, grid_H=la_H, grid_W=la_W).to(DEVICE)
            la_m.load_state_dict(base_model.state_dict())

        elif name == 'sttransformer':
            # Rebuild with LA grid dims — AdaptiveAvgPool handles variable H,W
            la_m = STTransformer(in_ch=1, hidden=64, n_heads=4, n_layers=2,
                                 dropout=0.1, grid_H=la_H, grid_W=la_W,
                                 seq_len=SEQ_LEN, n_patches=121).to(DEVICE)
            la_m.load_state_dict(base_model.state_dict())

        elif name == 'conv':
            la_m = ConvOnly(in_ch=_c1_in, hidden1=_c1_out, hidden2=_c2_out,
                            grid_H=la_H, grid_W=la_W).to(DEVICE)
            la_m.load_state_dict(base_model.state_dict())

        elif name == 'lstm':
            # LSTMOnly input dim is hardcoded to Chicago N nodes (_lstm_in=7310)
            # — cannot transfer to LA's different grid size; skip.
            print(f'  ⏭️  lstm: input dim tied to Chicago grid ({_lstm_in} nodes) '
                  f'vs LA ({la_N} nodes) — skipping')
            return None

        else:
            la_m = base_model
        la_m.eval()
        return la_m

    print('=' * 60)
    print('  LA CALIBRATION')
    print('=' * 60)

    la_results = {}
    for name, model in MODELS.items():
        print(f'\n── {name} ──')
        try:
            la_model = get_la_model(name, model)
            if la_model is None:
                continue
            m_raw, m_cal = calibrate_and_save(
                la_model, name, la_cal_dl, la_test_dl, 'la')
            la_results[name] = {'raw': m_raw, 'cal': m_cal}
            print(f'  ✅ LA artefacts saved → outputs/calibration/{name}/')
        except Exception as e:
            import traceback
            print(f'  ❌ Failed: {e}')
            traceback.print_exc()

    print('\n✅ LA calibration complete')

  LA CALIBRATION

── convlstm ──
  T_opt=0.1000   best_threshold=0.0582
  AUROC  raw=0.8373  cal=0.8373
  AUPRC  raw=0.1884  cal=0.1884
  F1     raw=0.0584     cal=0.2714
  ✅ LA artefacts saved → outputs/calibration/convlstm/

── sttransformer ──
  T_opt=20.0000   best_threshold=0.0337
  AUROC  raw=0.5517  cal=0.4483
  AUPRC  raw=0.0487  cal=0.0295
  F1     raw=0.0647     cal=0.0642
  ✅ LA artefacts saved → outputs/calibration/sttransformer/

── lstm ──
  ⏭️  lstm: input dim tied to Chicago grid (7310 nodes) vs LA (12474 nodes) — skipping

── conv ──
  T_opt=1.1631   best_threshold=0.1464
  AUROC  raw=0.9123  cal=0.9123
  AUPRC  raw=0.2515  cal=0.2515
  F1     raw=0.2749     cal=0.311
  ✅ LA artefacts saved → outputs/calibration/conv/

✅ LA calibration complete


## 📋 STEP 11 — Final Summary

In [15]:
print('=' * 70)
print('  FINAL SUMMARY — Calibrated metrics')
print('=' * 70)

for city, res_dict in [('Chicago', chi_results),
                        ('LA',      la_results if HAS_LA else {})]:
    if not res_dict:
        continue
    print(f'\n  ─── {city} ─────────────────────────────────────────────────')
    rows = []
    for name, res in res_dict.items():
        row = {'Model': name}
        row.update({k: res['cal'][k] for k in SHOW_COLS})
        rows.append(row)
    df = pd.DataFrame(rows).set_index('Model').sort_values('AUROC', ascending=False)
    print(df.to_string())

print('\n── Artefact locations ──')
for name in MODELS:
    for city in (['chicago'] + (['la'] if HAS_LA else [])):
        base = f'{BASE_DIR}/outputs/calibration/{name}/{city}'
        files = ['_proba_ts.npy', '_proba_cal.npy', '_true.npy', '_meta.json']
        ok = all(os.path.exists(base + f) for f in files)
        print(f'  {"✅" if ok else "❌"} {name}/{city}')

print('\n✅ All done — re-run CrimeHotspot_ModelComparison.ipynb to update plots.')

  FINAL SUMMARY — Calibrated metrics

  ─── Chicago ─────────────────────────────────────────────────
                AUROC   AUPRC      F1  Precision  Recall    PAI@100    PAI@200
Model                                                                         
lstm           0.9341  0.4308  0.4622     0.3760  0.5997  16.177099  16.177099
conv           0.9148  0.3546  0.3981     0.3449  0.4707  15.681900  15.186700
convlstm       0.8637  0.2968  0.3630     0.2937  0.4750  13.536000  13.536000
sttransformer  0.4393  0.0557  0.1142     0.0606  1.0000   0.165100   0.082500

  ─── LA ─────────────────────────────────────────────────
                AUROC   AUPRC      F1  Precision  Recall    PAI@100  PAI@200
Model                                                                       
conv           0.9123  0.2515  0.3110     0.2251  0.5030  18.404100  19.3092
convlstm       0.8373  0.1884  0.2714     0.2044  0.4038  16.895599  15.8396
sttransformer  0.4483  0.0295  0.0642     0.0331  1.0000